## MULTIPLE LINEAR REGRESSION

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# 1. LOAD DATA & EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
df = pd.read_csv('ToyotaCorolla - MLR.csv')

print("--- Dataset Shape ---")
print(df.shape)

print("\n--- Summary Statistics ---")
print(df.describe())

--- Dataset Shape ---
(1436, 11)

--- Summary Statistics ---
              Price    Age_08_04             KM           HP    Automatic  \
count   1436.000000  1436.000000    1436.000000  1436.000000  1436.000000   
mean   10730.824513    55.947075   68533.259749   101.502089     0.055710   
std     3626.964585    18.599988   37506.448872    14.981080     0.229441   
min     4350.000000     1.000000       1.000000    69.000000     0.000000   
25%     8450.000000    44.000000   43000.000000    90.000000     0.000000   
50%     9900.000000    61.000000   63389.500000   110.000000     0.000000   
75%    11950.000000    70.000000   87020.750000   110.000000     0.000000   
max    32500.000000    80.000000  243000.000000   192.000000     1.000000   

                cc        Doors  Cylinders        Gears      Weight  
count   1436.00000  1436.000000     1436.0  1436.000000  1436.00000  
mean    1576.85585     4.033426        4.0     5.026462  1072.45961  
std      424.38677     0.952677    

In [3]:
# Save target distribution plot
plt.figure(figsize=(8, 5))
sns.histplot(df['Price'], kde=True, color='skyblue')
plt.title('Distribution of Toyota Corolla Prices')
plt.xlabel('Price (EUROs)')
plt.ylabel('Count')
plt.savefig('price_distribution.png')
plt.close()

In [4]:
# DATA PREPROCESSING
# ==========================================
# 'Cylinders' has a standard deviation of 0 (every car has 4). 
# Constant columns add zero predictive power
df_cleaned = df.drop(columns=['Cylinders'])

# Handle Categorical Columns via One-Hot Encoding
df_cleaned = pd.get_dummies(df_cleaned, columns=['Fuel_Type'], drop_first=True)
print(df_cleaned)

      Price  Age_08_04     KM   HP  Automatic    cc  Doors  Gears  Weight  \
0     13500         23  46986   90          0  2000      3      5    1165   
1     13750         23  72937   90          0  2000      3      5    1165   
2     13950         24  41711   90          0  2000      3      5    1165   
3     14950         26  48000   90          0  2000      3      5    1165   
4     13750         30  38500   90          0  2000      3      5    1170   
...     ...        ...    ...  ...        ...   ...    ...    ...     ...   
1431   7500         69  20544   86          0  1300      3      5    1025   
1432  10845         72  19000   86          0  1300      3      5    1015   
1433   8500         71  17016   86          0  1300      3      5    1015   
1434   7250         70  16916   86          0  1300      3      5    1015   
1435   6950         76      1  110          0  1600      5      5    1114   

      Fuel_Type_Diesel  Fuel_Type_Petrol  
0                 True          

In [17]:
# Convert True/False flags to integer flags (1 or 0)
for col in ['Fuel_Type_Diesel', 'Fuel_Type_Petrol']:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype(int)
print(df_cleaned)

      Price  Age_08_04     KM   HP  Automatic    cc  Doors  Gears  Weight  \
0     13500         23  46986   90          0  2000      3      5    1165   
1     13750         23  72937   90          0  2000      3      5    1165   
2     13950         24  41711   90          0  2000      3      5    1165   
3     14950         26  48000   90          0  2000      3      5    1165   
4     13750         30  38500   90          0  2000      3      5    1170   
...     ...        ...    ...  ...        ...   ...    ...    ...     ...   
1431   7500         69  20544   86          0  1300      3      5    1025   
1432  10845         72  19000   86          0  1300      3      5    1015   
1433   8500         71  17016   86          0  1300      3      5    1015   
1434   7250         70  16916   86          0  1300      3      5    1015   
1435   6950         76      1  110          0  1600      5      5    1114   

      Fuel_Type_Diesel  Fuel_Type_Petrol  
0                    1          

In [7]:
# Separate Matrix of Features (X) and Target Vector (y)
X = df_cleaned.drop(columns=['Price'])
y = df_cleaned['Price']

# ---------------------------------------
# 2. DATA SPLIT (80% Train, 20% Test)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Helper function to compute test performance metrics
def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"[{model_name}] Test RMSE: {rmse:.2f} | Test MAE: {mae:.2f} | Test R2: {r2:.4f}")
    return rmse, mae, r2

In [9]:
#--- Model 1: the base line model ---
features_1 = ['Age_08_04', 'KM', 'HP', 'Weight']
X_tr1 = sm.add_constant(X_train[features_1])
X_te1 = sm.add_constant(X_test[features_1])
model_1 = sm.OLS(y_train, X_tr1).fit()

# --- Model 2: expansion for the base line model ---
features_2 = features_1 + ['cc', 'Doors', 'Gears', 'Automatic']
X_tr2 = sm.add_constant(X_train[features_2])
X_te2 = sm.add_constant(X_test[features_2])
model_2 = sm.OLS(y_train, X_tr2).fit()

# --- Model 3: Full model(including all the features here) ---
X_tr3 = sm.add_constant(X_train)
X_te3 = sm.add_constant(X_test)
model_3 = sm.OLS(y_train, X_tr3).fit()

In [10]:
print("\n--- Model Summaries (Training Results) ---")
print(f"Model 1 Adjusted R2: {model_1.rsquared_adj:.4f}")
print(f"Model 2 Adjusted R2: {model_2.rsquared_adj:.4f}")
print(f"Model 3 Adjusted R2: {model_3.rsquared_adj:.4f}")


--- Model Summaries (Training Results) ---
Model 1 Adjusted R2: 0.8638
Model 2 Adjusted R2: 0.8650
Model 3 Adjusted R2: 0.8691


In [14]:
# 4. MODEL EVALUATION ON TESTING DATASET
# ==========================================
print("\n--- Model Evaluation on Test Split ---")
pred_1 = model_1.predict(X_te1)
pred_2 = model_2.predict(X_te2)
pred_3 = model_3.predict(X_te3)

calculate_metrics(y_test, pred_1, "Model 1: Baseline model")
calculate_metrics(y_test, pred_2, "Model 2: Expansion for the baseline model")
calculate_metrics(y_test, pred_3, "Model 3: Full model")


--- Model Evaluation on Test Split ---
[Model 1: Baseline model] Test RMSE: 1411.85 | Test MAE: 1001.20 | Test R2: 0.8506
[Model 2: Expansion for the baseline model] Test RMSE: 1403.61 | Test MAE: 997.00 | Test R2: 0.8523
[Model 3: Full model] Test RMSE: 1484.27 | Test MAE: 990.89 | Test R2: 0.8349


(1484.2654153296671, 990.8872739194038, 0.8348888040611047)

In [16]:
# 5. REGULARIZATION (LASSO & RIDGE)
# ==========================================
# Scale data for meaningful regularization penalties
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Ridge Regression
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_train)
ridge_preds = ridge_reg.predict(X_test_scaled)

# Lasso Regression
lasso_reg = Lasso(alpha=1.0)
lasso_reg.fit(X_train_scaled, y_train)
lasso_preds = lasso_reg.predict(X_test_scaled)

print("\n--- Regularized Models Performance ---")
calculate_metrics(y_test, ridge_preds, "Ridge Regression")
calculate_metrics(y_test, lasso_preds, "Lasso Regression")


--- Regularized Models Performance ---
[Ridge Regression] Test RMSE: 1483.56 | Test MAE: 990.86 | Test R2: 0.8350
[Lasso Regression] Test RMSE: 1483.24 | Test MAE: 991.02 | Test R2: 0.8351


(1483.23568692162, 991.0174663805727, 0.8351178206648818)

**Model Comparison & Metrics Evaluation:

 Model 1 (baseline) yielded an Adjusted R2 of 0.864 on training data and an evaluation RMSE of 1,411.85.
 Model 2 (expansion of baseline) generated the lowest out-of-sample prediction error on the test dataset with an RMSE of 1,403.61.
 Model 3 (All-inclusive) demonstrated an adjusted R^2 of 0.869 on training data. However, adding highly collinear variables caused its generalization to drop slightly on testing data (RMSE: 1,484.27), a classic indicator of overfitting.

**Underlying Assumptions and Implications

Linearity (Fixed Depreciation vs. Real Life): The model assumes a car loses the exact same amount of cash value every single month or kilometer forever. In reality, cars lose value very quickly in their first few years and much slower as they get older. Because the model forces a straight line, it will artificially tank the price estimate of very old cars, guessing they are worth less than they actually are.

No Multicollinearity (Overlapping Features):
Features like engine size (cc), horsepower (HP), and car Weight are deeply connected—bigger engines naturally mean more power and heavier cars. Because these features tell the model the exact same story, the model gets confused trying to separate their individual impacts. As a result, it falsely labels variables like cc and Doors as "insignificant," even though they obviously affect a car's price.